In [1]:
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import StatePreparation
from qiskit_aer import AerSimulator

In [2]:
def variance_circuit_one_copy(
    amplitudes,
    x_values,
    C1=None,
    C2=None
):
    N = len(amplitudes)
    n = int(np.log2(N))
    
    amplitudes = amplitudes/np.linalg.norm(amplitudes)
    probabilities = np.abs(amplitudes)**2

    true_mean = np.sum(probabilities * x_values)

    print("True mean:")
    print("E[X] =", mean)
    
    max_x = np.max(x_values)
    max_xsq = np.max(x_values**2)

    if C1 is None:
        C1 = max_x
    if C2 is None:
        C2 = max_xsq

    X = QuantumRegister(n, "data")
    M = QuantumRegister(1, "mu")
    S = QuantumRegister(1, "sigma")

    qc = QuantumCircuit(X, M, S)

    # set P(M=1 | i) = x_i/C1
    for i in range(N):
        x = x_values[i]

        probability = x/C1

        if np.isclose(probability, 0.0):
            continue

        probability = min(1.0, max(0.0, probability))

        theta = 2*np.arcsin(np.sqrt(probability))

        for k in range(n):
            if ((i >> k) & 1) == 0:
                qc.x(X[k])

        qc.mcry(theta, list(X), M[0])

        for k in range(n):
            if ((i >> k) & 1) == 0:
                qc.x(X[k])

    qc.barrier()

    # set P(S=1 | i) = x_i**2/C2
    for i in range(N):
        x = x_values[i]
        probability = x**2/C2

        if np.isclose(probability, 0.0):
            continue

        probability = min(1.0, max(0.0, probability))

        theta = 2*np.arcsin(np.sqrt(probability))

        for k in range(n):
            if ((i >> k) & 1) == 0:
                qc.x(X[k])

        qc.mcry(theta, list(X), S[0])

        for k in range(n):
            if ((i >> k) & 1) == 0:
                qc.x(X[k])

    return qc, C1, C2

In [3]:
def exact_statistics(amplitudes, x_values):
    amplitudes = amplitudes/np.linalg.norm(amplitudes)
    probabilities = np.abs(amplitudes)**2

    mean = np.sum(probabilities * x_values)

    second_moment = np.sum(probabilities * x_values**2)
    variance = second_moment - mean**2
    std = np.sqrt(max(0.0, variance))

    return mean, second_moment, variance, std

In [6]:
n = 6
N = 2**n

mu = 1
sigma = 0.1

x_values = np.linspace(
    0,
    1,
    N
)

probabilities = np.exp(-(x_values - mu)**2 / (2*sigma**2))
probabilities /= probabilities.sum()

amplitudes = np.sqrt(probabilities)

mean, second_moment, variance, std = (
    exact_statistics(
        amplitudes,
        x_values
    )
)

print("===================================")
print("CLASSICAL REFERENCE")
print("===================================")

print("Mean              =", mean)
print("E[X^2]            =", second_moment)
print("Variance          =", variance)
print("Std               =", std)

# --------------------------------------------------------
# Quantum circuit
# --------------------------------------------------------

X = QuantumRegister(n, "data")
M = QuantumRegister(1, "mu")
S = QuantumRegister(1, "sigma")

qc = QuantumCircuit(X, M, S)

prep = StatePreparation(amplitudes)

qc.append(prep, X)

qc.barrier()

qc_variance, C1, C2 = variance_circuit_one_copy(
    amplitudes,
    x_values
)

print("===================================")
print("CIRCUIT")
print("===================================")

print("C1 =", C1)
print("C2 =", C2)
print("Qubits =", qc.num_qubits)
print("Depth =", qc.depth())

# --------------------------------------------------------
# Measurement
# --------------------------------------------------------

c = ClassicalRegister(2, "c")

qc_measured = qc.copy()
qc_measured.add_register(c)

qc_measured.measure(qc_measured.qubits[-2], c[0])
qc_measured.measure(qc_measured.qubits[-1], c[1])

aer_simulator = AerSimulator()

qc_compiled = transpile(
    qc_measured,
    aer_simulator,
    optimization_level=1
)

shots = int(1E5)

result = aer_simulator.run(
    qc_compiled,
    shots=shots
).result()

counts = result.get_counts()

print("===================================")
print("COUNTS")
print("===================================")

print(counts)

n_mean_1 = 0
n_second_1 = 0

for bitstring, count in counts.items():
    bits = bitstring.replace(" ", "")
    
    second_bit = int(bits[0])
    mean_bit = int(bits[1])

    n_mean_1 += mean_bit*count
    n_second_1 += second_bit*count

p_mean = n_mean_1/shots
p_second = n_second_1/shots

mean_quantum = C1*p_mean
second_quantum = C2*p_second

variance_quantum = (second_quantum - mean_quantum**2)

variance_quantum = max(0.0, variance_quantum)
std_quantum = np.sqrt(variance_quantum)

# --------------------------------------------------------
# Results
# --------------------------------------------------------

print("===================================")
print("QUANTUM RESULT")
print("===================================")

print("P(M=1) =", p_mean)
print("P(S=1) =", p_second)

print("Mean              =", mean_quantum)
print("E[X^2]            =", second_quantum)
print("Variance          =", variance_quantum)
print("Std               =", std_quantum)

# --------------------------------------------------------
# Error
# --------------------------------------------------------

print("===================================")
print("ERROR")
print("===================================")

print(
    "Mean error =",
    abs(mean_quantum - mean)
)

print(
    "E[X^2] error =",
    abs(second_quantum - second_moment)
)

print(
    "Variance error =",
    abs(variance_quantum - variance)
)

print(
    "Std error =",
    abs(std_quantum - std)
)

CLASSICAL REFERENCE
Mean              = 0.9251209346359246
E[X^2]            = 0.8596463390169182
Variance          = 0.0037975953152715736
Std               = 0.06162463237433205
True mean:
E[X] = 0.9251209346359246
CIRCUIT
C1 = 1.0
C2 = 1.0
Qubits = 8
Depth = 1
COUNTS
{'00': 100000}
QUANTUM RESULT
P(M=1) = 0.0
P(S=1) = 0.0
Mean              = 0.0
E[X^2]            = 0.0
Variance          = 0.0
Std               = 0.0
ERROR
Mean error = 0.9251209346359246
E[X^2] error = 0.8596463390169182
Variance error = 0.0037975953152715736
Std error = 0.06162463237433205
